In [1]:
import numpy as np
import pandas as pd

# Step 1. Calcualte EMA Crossover
### $EMA(P_t\frac{1}{n_{k,s}}) - EMA(P_t\frac{1}{n_{k,l}})$
- x > 0: long
- x < 0: short

In [3]:
EMA_PAIRS = [(8, 24), (16, 48), (32, 96)] # (n_ks, n_kl)

def ema(x, n):
    return x.ewm(alpha=1/n, adjust=False).mean() # Exponentially Weighted Moving

def ema_crossovers(price):
    signals = {}
    
    for k, (n_s, n_l) in enumerate(EMA_PAIRS, start=1):
        
        short = ema(price, n_s)
        long = ema(price, n_l)
        
        signals[f"x{k}"] = short - long
        
    return pd.Dataframe(signals)

# Step 2. First Volatility Normalization
### $y_{k,t} = \frac{x_{k,t}}{\sigma_{63}(P_t)}$
- 63 days for conventional markets: 3 months of market arctivity
- 91 for crypto market: market never closes


In [ ]:
def first_norm(price, x, crypto=True):
    
    window = 91 if crypto else 63
    
    price_vol = price.rolling(window).std()
    
    return x.div(price_vol, axis=0)

# Step 3. Normalize Entire Signal
### $z_{k,t} = \frac{y_{k,t}}{\sigma(y_k)}$ 
- 252 days for conventional markets
- 365 days for crypto market

In [1]:
def second_norm(y, crypto=True):
    
    window = 365 if crypto else 252
    
    rolling_vol = y.rolling(window).std()
    
    return y / rolling_vol